## Task 9: Test FOV Inference & Submission Generation

Run the fine-tuned Cellpose model on the 4 test FOVs, assign mRNA spots to cells via pixel lookup, and generate `submission.csv` with integer cluster_ids (0 = background).

In [ ]:
import sys, os
sys.path.insert(0, '..')
import numpy as np
import pandas as pd

DATA_ROOT = "/scratch/pl2820/competition"
test_spots   = pd.read_csv(f"{DATA_ROOT}/test_spots.csv")
sub_template = pd.read_csv(f"{DATA_ROOT}/sample_submission.csv")
meta         = pd.read_csv(f"{DATA_ROOT}/reference/fov_metadata.csv").set_index("fov")

TEST_FOVS = ["FOV_A", "FOV_B", "FOV_C", "FOV_D"]

print(f"Test spots shape  : {test_spots.shape}")
print(f"Template rows     : {len(sub_template)}")
print("\nSpots per FOV:")
print(test_spots["fov"].value_counts())

### Step 1: Load segmentation model

In [ ]:
from cellpose import models as cp_models

model_path = "models/cellpose_finetuned"
if os.path.exists(model_path):
    seg_model = cp_models.CellposeModel(gpu=True, pretrained_model=model_path)
    print(f"Loaded fine-tuned model from: {model_path}")
    def run_segmentation(image):
        masks, _, _ = seg_model.eval(image, diameter=30, channels=[1, 2])
        return masks
else:
    print("Fine-tuned model not found — falling back to pretrained nuclei")
    seg_model = cp_models.CellposeModel(gpu=False, model_type="cyto2")
    def run_segmentation(image):
        masks, _, _ = seg_model.eval(image, diameter=30, channels=[1, 2])
        return masks


### Step 2: Run segmentation on each test FOV

Spot assignment via direct pixel lookup: `cluster_id = mask[pixel_y, pixel_x]`

In [ ]:
from src.io import load_fov_images

all_cluster_ids = {}   # spot_id -> integer cluster_id (0 = background)

for fov_name in TEST_FOVS:
    fov_dir = f"{DATA_ROOT}/test/{fov_name}"
    if not os.path.exists(fov_dir):
        print(f"  Missing FOV dir: {fov_dir}")
        continue

    dapi, polyt = load_fov_images(fov_dir)
    masks = run_segmentation(np.stack([polyt[2], dapi[2]], axis=0))  # integer array, 0=background, 1..N=cells
    print(f"{fov_name}: {masks.max()} cells detected")

    fov_spots = test_spots[test_spots["fov"] == fov_name].copy()
    # Use pre-computed image_row / image_col columns (correct MERFISH convention):
    #   image_row = 2048 - (global_x - fov_x) / pixel_size  (x-axis is flipped)
    #   image_col = (global_y - fov_y) / pixel_size
    rows = np.clip(fov_spots["image_row"].values.astype(int), 0, 2047)
    cols = np.clip(fov_spots["image_col"].values.astype(int), 0, 2047)
    cluster_ids = masks[rows, cols]  # direct lookup: 0=background, 1..N=cell

    for spot_id, cid in zip(fov_spots["spot_id"].values, cluster_ids):
        all_cluster_ids[spot_id] = int(cid)  # 0 means background

    bg_frac = (cluster_ids == 0).mean()
    print(f"  {len(fov_spots):,} spots, {bg_frac:.1%} background")

print(f"\nTotal spots assigned: {len(all_cluster_ids):,}")

### Step 3: Build and validate submission.csv

In [ ]:
sub = sub_template.copy()
sub["cluster_id"] = sub["spot_id"].map(all_cluster_ids)

null_count = sub["cluster_id"].isna().sum()
print(f"Null cluster_ids: {null_count}")
if null_count > 0:
    print("  WARNING: some spots not found — filling with 0 (background)")
    sub["cluster_id"] = sub["cluster_id"].fillna(0)

sub["cluster_id"] = sub["cluster_id"].astype(int)

assert len(sub) == len(sub_template), f"Row count mismatch: {len(sub)} != {len(sub_template)}"
assert list(sub.columns) == ["spot_id", "fov", "cluster_id"], f"Wrong columns: {list(sub.columns)}"
assert sub["cluster_id"].notna().all(), "Null cluster_ids found!"

sub.to_csv("../submission.csv", index=False)
print(f"\nSaved submission.csv — {len(sub):,} rows")
print(sub["cluster_id"].value_counts().head(10))
print(sub.head())
